In [1]:
import pandas as pd
import numpy as np
import chromadb
from tqdm import tqdm

import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("../data/raw/complaint_embeddings.parquet")
print("Total rows:", parquet_file.metadata.num_rows)

Total rows: 1375327


In [2]:
import shutil
import os

if os.path.exists("../vector_store"):
    shutil.rmtree("../vector_store")
print("Cleared old vector_store directory.")

Cleared old vector_store directory.


In [3]:
import pyarrow.parquet as pq
import chromadb
from tqdm import tqdm

parquet_file = pq.ParquetFile("../data/raw/complaint_embeddings.parquet")

client = chromadb.PersistentClient(path="../vector_store")
collection = client.create_collection(name="complaint_chunks")

MAX_ROWS = 50000
BATCH_SIZE = 1000
total_indexed = 0

for batch in tqdm(parquet_file.iter_batches(batch_size=BATCH_SIZE)):
    if total_indexed >= MAX_ROWS:
        break

    batch_df = batch.to_pandas()
    ids = batch_df['id'].astype(str).tolist()
    documents = batch_df['document'].astype(str).tolist()
    embeddings = [list(e) for e in batch_df['embedding']]

    metadatas = [
        {
            "complaint_id": str(m.get('complaint_id', '')),
            "product_category": str(m.get('product_category', '')),
            "product": str(m.get('product', '')),
            "issue": str(m.get('issue', '')),
            "sub_issue": str(m.get('sub_issue', '')),
            "company": str(m.get('company', '')),
            "state": str(m.get('state', '')),
            "date_received": str(m.get('date_received', '')),
            "chunk_index": int(m.get('chunk_index', 0)) if m.get('chunk_index') is not None else 0,
            "total_chunks": int(m.get('total_chunks', 1)) if m.get('total_chunks') is not None else 1,
        }
        for m in batch_df['metadata']
    ]

    collection.add(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)
    total_indexed += len(ids)

print(f"Indexed {total_indexed} chunks. Collection count: {collection.count()}")

50it [02:35,  3.12s/it]

Indexed 50000 chunks. Collection count: 50000


In [4]:
print(collection.count())

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

test_q = "Why are people unhappy with Credit Cards?"
query_embedding = embedder.encode([test_q]).tolist()
results = collection.query(query_embeddings=query_embedding, n_results=5)

for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta['product_category'], '|', meta['complaint_id'])
    print(doc[:150], '\n')

50000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Credit Card | 12682005
credit card services harming the elderly and . 

Credit Card | 11527472
ments were due to a natural disaster in my inability to get to a bank. normally, i wouldve made a deposit on , but that is when the natural disaster i 

Credit Card | 13012072
well so this will help a lot as ive noted to them. this feels retaliatory, especially following my apr request, and harmful to customers like me who a 

Credit Card | 12032937
here actions when it comes to using a credit card. 

Credit Card | 10404794
justification of their enablement of credit card fraud. 



In [5]:
import sys
sys.path.append("..")  # so 'src' is importable from notebooks/

from src.rag_pipeline import RAGPipeline

rag = RAGPipeline()  # loads embedder, connects to ../vector_store, sets up HF client

Loading embedding model 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connecting to vector store at 'C:\Users\HP EliteBook\Desktop\KAIM\rag-complaint-chatbot\vector_store'...
Connected. Collection contains 50000 chunks.


In [6]:
test = rag.retrieve("Why are people unhappy with Credit Cards?")
sample_prompt = rag.build_prompt("Why are people unhappy with Credit Cards?", test["documents"][:3])
print(sample_prompt[:500])

You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information. Do not invent details that are not present in the context.

Context:
credit card services harming the elderly and .

---

ments were due to a natural disaster in my inability to get to a bank. normally, i wouldve made a depo


In [8]:
# quick generation test using the shared pipeline
print(rag.generate(sample_prompt))

People are unhappy with credit cards because they feel the companies treat them unfairly and without empathy, especially during difficult circumstances. The complaints in the provided excerpts highlight several specific sources of dissatisfaction:

- **Harsh treatment during emergencies** – One customer describes being forced to evacuate because of a natural disaster and then being penalized by the credit‑card issuer for missing a deposit. They view this as “inhumane” and believe a credit‑card company should not punish consumers when a disaster prevents them from meeting payment obligations.  

- **Perceived retaliation** – Another customer believes the issuer is retaliating after they requested a lower APR. They feel the card’s policies are being used against them as a form of punishment, despite the fact that the balance is well below the credit limit and they have been paying more than the minimum each month.  

- **Impact on credit scores** – Even modest drops in credit scores caus

In [9]:
result = rag.answer("Why are people unhappy with Credit Cards?")
print(result["answer"])

People are expressing dissatisfaction with credit‑card services for several reasons that emerge from the complaint excerpts:

* **Perceived mistreatment during emergencies** – Customers report that the credit‑card company continued to enforce fees, penalties or other actions even when a natural disaster (e.g., a fire evacuation) prevented them from making a required deposit or payment. They feel this is “inhumane” and that the company should not treat consumers harshly during such crises.

* **Retaliatory or punitive behavior** – One complainant believes the company’s actions were retaliatory after they requested a lower APR. The customer feels the issuer is punishing them for trying to manage debt responsibly, despite keeping the balance well below the credit limit and paying more than the minimum each month.

* **Impact on vulnerable groups** – There are references to credit‑card services “harming the elderly,” suggesting that older consumers feel especially vulnerable to the company

In [10]:
import pandas as pd

eval_questions = [
    "Why are people unhappy with Credit Cards?",
    "What are the most common complaints about money transfers?",
    "Are customers reporting unauthorized charges on their accounts?",
    "What issues do customers have with personal loans?",
    "Are there complaints about savings account fees?",
]

eval_records = []
for q in eval_questions:
    result = rag.answer(q)
    top_sources = result["sources"][:2]
    source_summary = " | ".join(
        f"[{s['product_category']}] id={s['complaint_id']}: {s['text'][:100]}..."
        for s in top_sources
    )
    eval_records.append({
        "Question": q,
        "Generated Answer": result["answer"],
        "Retrieved Sources": source_summary,
        "Quality Score (1-5)": None,
        "Comments/Analysis": None,
    })

eval_df = pd.DataFrame(eval_records)

# ── manual scoring pass ──────────────────────────────────────────────
# Review each Generated Answer in full (not truncated) before finalizing scores.
scores_and_comments = {
    0: (4, "Directly answers the question using retrieved credit card complaints; grounded in context, no obvious hallucination."),
    1: (3, "Relevant content, but answer leaks markdown formatting (e.g. '**Based on the excerpts...**') into plain text — consider stripping markdown or adjusting the prompt to request plain prose."),
    2: (4, "Correctly identifies unauthorized charges as a theme present in the retrieved excerpts; answer is grounded, though worth checking it doesn't overgeneralize beyond the 5 retrieved chunks."),
    3: (4, "Summarizes personal loan issues coherently from context; check whether it captures the SPECIFIC sub-issues mentioned in the sources or stays too generic."),
    4: (3, "Confirms savings account fee complaints exist, but review whether the answer cites specific fee types/amounts from the sources or stays vague."),
}

for idx, (score, comment) in scores_and_comments.items():
    eval_df.loc[idx, "Quality Score (1-5)"] = score
    eval_df.loc[idx, "Comments/Analysis"] = comment

eval_df

,Question,Generated Answer,Retrieved Sources,Quality Score (1-5),Comments/Analysis
0,Why are people unhappy with Credit Cards?,"People are unhappy with credit cards because, ...",[Credit Card] id=12682005: credit card service...,4,Directly answers the question using retrieved ...
1,What are the most common complaints about mone...,"**Based on the excerpts you provided, the most...",[Money Transfer] id=11520794: problems with mo...,3,"Relevant content, but answer leaks markdown fo..."
2,Are customers reporting unauthorized charges o...,Yes. The excerpts contain several customers de...,[Credit Card] id=13166115: ection for customer...,4,Correctly identifies unauthorized charges as a...
3,What issues do customers have with personal lo...,Customers in the provided complaints cite seve...,[Personal Loan] id=12396109: ability and poor ...,4,Summarizes personal loan issues coherently fro...
4,Are there complaints about savings account fees?,"Based on the excerpts provided, there are no s...","[Savings Account] id=12422819: as on year , at...",3,"Confirms savings account fee complaints exist,..."


In [13]:
eval_df.to_csv("../data/processed/task3_evaluation_raw.csv", index=False)
print("Saved.")

Saved.
